In [1]:
!pip install -q marker-pdf PyMuPDF torch
import torch
print(f"CUDA: {torch.cuda.is_available()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1

In [2]:
from google.colab import files
from pathlib import Path

INPUT_DIR = Path('/content/gdrive/MyDrive/docling_input')
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Chọn n file PDF (giữ Ctrl/Cmd để chọn nhiều):")
uploaded = files.upload()

for name, data in uploaded.items():
    if name.endswith('.pdf'):
        (INPUT_DIR / name).write_bytes(data)
        print(f"  Saved: {name}")

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f"\n{len(pdfs)} PDFs ready:")
for p in pdfs:
    print(f"  {p.name[:55]}  ({p.stat().st_size//1024} KB)")

Chọn n file PDF (giữ Ctrl/Cmd để chọn nhiều):


Saving Macroeconomic Accounting and Analysis IMF.pdf to Macroeconomic Accounting and Analysis IMF.pdf
  Saved: Macroeconomic Accounting and Analysis IMF.pdf

1 PDFs ready:
  Macroeconomic Accounting and Analysis IMF.pdf  (12833 KB)


In [ ]:
import time, gc, fitz, torch
from pathlib import Path

# Marker API (math/table-aware PDF -> Markdown)
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

INPUT_DIR = Path('/content/gdrive/MyDrive/docling_input')
MD_OUT    = Path('/content/gdrive/MyDrive/docling_output')
MD_OUT.mkdir(parents=True, exist_ok=True)

# --- FIX: build converter MỘT lần, tái dùng qua nhiều batch (tránh reload model + OOM tích luỹ) ---
if 'converter' not in globals():
    print("Tải model Marker (chỉ chạy lần đầu)...")
    model_dict = create_model_dict()
    converter  = PdfConverter(artifact_dict=model_dict)
else:
    print("Tái dùng converter đã nạp trong RAM.")

# --- FIX: overlap giữa các chunk để giữ ngữ cảnh xuyên trang (bảng/heading không bị cắt đôi) ---
PAGE_CHUNK = 40          # số trang mỗi lần convert (tăng từ 30 -> ít ranh giới hơn)
OVERLAP    = 2           # số trang chồng lấn giữa 2 chunk liên tiếp
STEP       = PAGE_CHUNK - OVERLAP

def _norm(block: str) -> str:
    """Chuẩn hoá whitespace để so khớp block trùng ở vùng overlap."""
    return " ".join(block.split())

pdf_files = sorted(INPUT_DIR.glob('*.pdf'))

for src in pdf_files:
    out_path = MD_OUT / f"{src.stem}.md"
    print(f"\n=== {src.name} ===")

    try:
        doc = fitz.open(src)
    except Exception as e:
        print(f"  ERR open: {e}")
        continue

    total_pages = len(doc)
    print(f"  Pages: {total_pages} | chunk={PAGE_CHUNK} overlap={OVERLAP}")

    if out_path.exists():
        out_path.unlink()

    seen = set()             # block đã ghi (đã chuẩn hoá) -> dùng để cắt vùng overlap đầu chunk
    failed_ranges = []       # FIX: log range fail thay vì nuốt thầm lặng

    for chunk_start in range(1, total_pages + 1, STEP):
        chunk_end = min(chunk_start + PAGE_CHUNK - 1, total_pages)
        temp_pdf  = Path(f'/tmp/_{src.stem}_{chunk_start}.pdf')

        try:
            # Cắt nhỏ PDF bằng PyMuPDF
            chunk_doc = fitz.open()
            chunk_doc.insert_pdf(doc, from_page=chunk_start-1, to_page=chunk_end-1)
            chunk_doc.save(temp_pdf)
            chunk_doc.close()

            rendered = converter(str(temp_pdf))
            full_text, _, _ = text_from_rendered(rendered)
            blocks = full_text.strip().split("\n\n")

            # FIX: cắt run block trùng ở ĐẦU chunk (vùng overlap đã có ở chunk trước)
            i0 = 0
            if chunk_start > 1:
                while i0 < len(blocks) and _norm(blocks[i0]) in seen:
                    i0 += 1
            kept = blocks[i0:]
            for b in kept:
                seen.add(_norm(b))

            md_text = "\n\n".join(kept).strip()
            if md_text:
                with open(out_path, 'a', encoding='utf-8') as fo:
                    fo.write(f"\n\n\n\n{md_text}")

            print(f"  chunk {chunk_start}-{chunk_end} OK  (+{len(kept)} blocks, bỏ {i0} overlap)")

            del rendered, full_text, blocks, kept
            gc.collect()
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"  chunk {chunk_start}-{chunk_end} FAIL: {type(e).__name__} - {e}")
            failed_ranges.append((chunk_start, chunk_end))
        finally:
            if temp_pdf.exists():
                temp_pdf.unlink()

        if chunk_end >= total_pages:
            break

    doc.close()
    gc.collect()
    torch.cuda.empty_cache()

    if failed_ranges:
        print(f"  ⚠ FAIL ranges (chạy lại các trang này): {failed_ranges}")
    else:
        print(f"  Done: {out_path.name}  (không có chunk lỗi)")

print(f"\nAll done! Output: {MD_OUT}")

In [4]:
from google.colab import files
from pathlib import Path
MD_OUT = Path('/content/gdrive/MyDrive/docling_output')
for md in sorted(MD_OUT.glob('*.md')):
    print(f"  {md.name} ({md.stat().st_size//1024} KB)")
    files.download(str(md))

  Macroeconomic Accounting and Analysis IMF.md (823 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, gc, torch
from pathlib import Path

# Dọn input cũ + GPU cache GIỮA CÁC BATCH.
# FIX: GIỮ converter trong RAM để batch sau tái dùng (không reload model).
#       (bỏ vòng purge 'docling' cũ — vô tác dụng vì module của Marker là 'marker.*')
INPUT_DIR = Path('/content/gdrive/MyDrive/docling_input')
shutil.rmtree(INPUT_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

gc.collect()
torch.cuda.empty_cache()

print("Đã dọn input + GPU cache. Converter vẫn nạp sẵn trong RAM.")
print("Tiếp: chạy lại cell UPLOAD -> cell CONVERT -> cell DOWNLOAD cho batch mới.")